In [5]:
import sys
sys.path.append("..")  # so `utils` resolves the same way the other notebooks expect

from pathlib import Path
from utils import db
from utils.scraping import tfl

DATA_DIR = Path("data/tmp")

# 1. scrape: pull the latest TfL station data feed and unpack it
tfl.download_station_data(DATA_DIR)
feed_version = tfl.read_feed_version(DATA_DIR)

# 2. update: diff it against the db and version only what changed
conn = db.get_connection()
load_id = db.start_load(conn, source="tfl_stationdata", feed_version=feed_version)

stations_result = db.ingest_csv(
    conn, DATA_DIR / "Stations.csv", table="stations",
    key_cols=["UniqueId"],
    attr_cols=["Name", "FareZones", "HubNaptanCode", "Wifi", "OutsideStationUniqueId",
               "BlueBadgeCarParking", "BlueBadgeCarParkSpaces", "TaxiRanksOutsideStation",
               "MainBusInterchange", "PierInterchange", "NationalRailInterchange",
               "AirportInterchange", "EmiratesAirLineInterchange"],
    load_id=load_id,
)

station_points_result = db.ingest_csv(
    conn, DATA_DIR / "StationPoints.csv", table="station_points",
    key_cols=["UniqueId"],
    attr_cols=["StationUniqueId", "AreaName", "AreaId", "Level", "Lat", "Lon", "FriendlyName"],
    load_id=load_id,
)

conn.commit()
print("feed_version:", feed_version)
print("load_id:", load_id)
print("stations:", stations_result)
print("station_points:", station_points_result)


feed_version: 2026-08-03T09:14+00:00
load_id: 3
stations: {'inserted': 0, 'closed': 0}
station_points: {'inserted': 0, 'closed': 0}


In [ ]:
# average position per station, ignoring bus areas.
# LIKE is case-insensitive in SQLite, so this catches Bus, BUS, busp and BusEF alike;
# COALESCE keeps points with a blank AreaName instead of silently dropping them.
conn.executescript("""
    CREATE VIEW IF NOT EXISTS station_coordinates AS
    SELECT
        StationUniqueId,
        AVG(Lat)  AS latitude,
        AVG(Lon)  AS longitude,
        COUNT(*)  AS n_points
    FROM current_station_points
    WHERE COALESCE(AreaName, '') NOT LIKE '%Bus%'
    GROUP BY StationUniqueId;
""")
conn.commit()

pl.read_database("SELECT * FROM station_coordinates", conn)


StationUniqueId,latitude,longitude,n_points
str,f64,f64,i64
"""910GACTNCTL""",51.508676,-0.26301,6
"""910GACTONML""",51.516941,-0.268227,5
"""910GANERLEY""",51.412432,-0.065567,6
"""910GBARKRIV""",51.519995,0.116589,5
"""910GBCKNHMH""",51.424701,-0.016238,6
…,…,…,…
"""HUBWWA""",51.490133,0.0690889,10
"""HUBZCW""",51.497993,-0.049676,9
"""HUBZFD""",51.520336,-0.104473,21


In [ ]:
# station names and attributes joined onto the averaged coordinates.
# Naming the columns keeps the SCD2 bookkeeping (row_id, valid_from_load, ...) out of the result.
pl.read_database("""
    SELECT
        s.UniqueId,
        s.Name,
        sc.latitude,
        sc.longitude,
        sc.n_points,
        s.FareZones,
        s.Wifi
    FROM station_coordinates sc
    LEFT JOIN current_stations s
        ON sc.StationUniqueId = s.UniqueId
    ORDER BY s.Name
""", conn)


In [9]:
pl.read_database("SELECT type, name FROM sqlite_master WHERE type IN ('table','view')", conn)


type,name
str,str
"""table""","""loads"""
"""table""","""stations"""
"""view""","""current_stations"""
"""table""","""station_points"""
"""view""","""current_station_points"""
"""view""","""station_coordinates"""


In [11]:
pl.read_database("SELECT * FROM stations LIMIT 10", conn)

row_id,UniqueId,Name,FareZones,HubNaptanCode,Wifi,OutsideStationUniqueId,BlueBadgeCarParking,BlueBadgeCarParkSpaces,TaxiRanksOutsideStation,MainBusInterchange,PierInterchange,NationalRailInterchange,AirportInterchange,EmiratesAirLineInterchange,valid_from_load,valid_to_load
i64,str,str,str,str,str,str,str,str,str,null,null,null,null,null,i64,null
1,"""HUBABW""","""Abbey Wood""","""4""","""HUBABW""","""false""","""HUBABW-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
2,"""910GACTNCTL""","""Acton Central""","""3""",null,"""true""","""910GACTNCTL-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
3,"""910GACTONML""","""Acton Main Line""","""3""",null,"""false""","""910GACTONML-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
4,"""910GANERLEY""","""Anerley""","""4""",null,"""true""","""910GANERLEY-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
5,"""910GBCKNHMH""","""Beckenham Hill""","""4""",null,"""false""","""910GBCKNHMH-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
6,"""910GBELNGHM""","""Bellingham""","""3""",null,"""false""","""910GBELNGHM-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
7,"""910GBHILLPK""","""Bush Hill Park""","""5""",null,"""true""","""910GBHILLPK-Outside""","""true""","""2""","""false""",null,null,null,null,null,1,null
8,"""910GBICKLEY""","""Bickley""","""5""",null,"""false""","""910GBICKLEY-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
9,"""910GBNHAM""","""Burnham""","""Outside""",null,"""false""","""910GBNHAM-Outside""","""false""",null,"""false""",null,null,null,null,null,1,null
